In [18]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path

# --- Path resolution (same convention as the pilot notebook) ---
_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(f"Could not find data/raw/DHS_Districts from cwd {_cwd}.")

MICRO_DIR = REPO_ROOT / "data" / "raw" / "DHS_microdata"
CROSSWALK_DIR = REPO_ROOT / "data" / "processed" / "district_crosswalks"

IR4_PATH = MICRO_DIR / "NFHS4_2015-16_IndividualRecode" / "IAIR74FL.DTA"
IR5_PATH = MICRO_DIR / "NFHS5_2019-21_IndividualRecode" / "IAIR7EFL.DTA"
BR4_PATH = MICRO_DIR / "NFHS4_2015-16_BirthsRecode" / "IABR74FL.DTA"
BR5_PATH = MICRO_DIR / "NFHS5_2019-21_BirthsRecode" / "IABR7EFL.DTA"

for p in [IR4_PATH, IR5_PATH, BR4_PATH, BR5_PATH]:
    print(p, "->", p.exists())

# --- Confirm column availability before reading full files ---
def peek(path, usecols=None, row_limit=5):
    df, meta = pyreadstat.read_dta(str(path), usecols=usecols, row_limit=row_limit)
    return df, meta

_, ir4_meta = peek(IR4_PATH)
_, ir5_meta = peek(IR5_PATH)
_, br4_meta = peek(BR4_PATH)
_, br5_meta = peek(BR5_PATH)

ir4_cols, ir5_cols = set(ir4_meta.column_names), set(ir5_meta.column_names)
br4_cols, br5_cols = set(br4_meta.column_names), set(br5_meta.column_names)

# --- Load full IR files: district code, weights, timing/age vars, controls ---
# v011 = woman's date of birth (CMC) -- needed to reconstruct her age in any
# historical year for the exposure/denominator segment. v012 is age at
# interview only and can't reconstruct historical age as precisely.
IR4_VARS = [c for c in ["caseid", "sdistri", "v005", "v008", "v011", "v025", "v106", "v190", "v104"] if c in ir4_cols]
IR5_VARS = [c for c in ["caseid", "sdist",   "v005", "v008", "v011", "v025", "v106", "v190", "v104"] if c in ir5_cols]

ir4, _ = pyreadstat.read_dta(str(IR4_PATH), usecols=IR4_VARS, apply_value_formats=False)
ir5, _ = pyreadstat.read_dta(str(IR5_PATH), usecols=IR5_VARS, apply_value_formats=False)

ir4 = ir4.rename(columns={"sdistri": "district_code_raw"})
ir5 = ir5.rename(columns={"sdist": "district_code_raw"})
print("IR4 shape:", ir4.shape, "| IR5 shape:", ir5.shape)

# --- Full crosswalk, all 575 stable districts (no pilot subsetting) ---
crosswalk = pd.read_csv(CROSSWALK_DIR / "nfhs4_nfhs5_stable_district_crosswalk.csv")
print("Crosswalk columns:", crosswalk.columns.tolist())
print("Stable districts in crosswalk:", crosswalk["district_code"].nunique())

nfhs4_to_stable = dict(zip(crosswalk["district_code"], crosswalk["district_code"]))
nfhs5_to_stable = dict(zip(crosswalk["district_code_nfhs5"], crosswalk["district_code"]))

ir4["stable_district_id"] = ir4["district_code_raw"].map(nfhs4_to_stable)
ir5["stable_district_id"] = ir5["district_code_raw"].map(nfhs5_to_stable)

# --- Coverage diagnostics: confirm mapping worked for the full sample ---
ir4_unmapped = ir4["stable_district_id"].isna().sum()
ir5_unmapped = ir5["stable_district_id"].isna().sum()
print(f"IR4: {ir4_unmapped} rows ({ir4_unmapped/len(ir4):.1%}) unmapped to a stable district")
print(f"IR5: {ir5_unmapped} rows ({ir5_unmapped/len(ir5):.1%}) unmapped to a stable district")
print(f"IR4 stable districts represented: {ir4['stable_district_id'].nunique()} / {crosswalk['district_code'].nunique()}")
print(f"IR5 stable districts represented: {ir5['stable_district_id'].nunique()} / {crosswalk['district_code'].nunique()}")

# Restrict to mappable (stable-district) women only -- unmapped rows are
# split-parent districts already excluded by design from the causal panel
ir4_full = ir4[ir4["stable_district_id"].notna()].copy()
ir5_full = ir5[ir5["stable_district_id"].notna()].copy()
print("Final scoped IR4:", ir4_full.shape, "| Final scoped IR5:", ir5_full.shape)

/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS4_2015-16_IndividualRecode/IAIR74FL.DTA -> True
/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS5_2019-21_IndividualRecode/IAIR7EFL.DTA -> True
/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS4_2015-16_BirthsRecode/IABR74FL.DTA -> True
/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS5_2019-21_BirthsRecode/IABR7EFL.DTA -> True
IR4 shape: (699686, 9) | IR5 shape: (724115, 9)
Crosswalk columns: ['district_code', 'district_name_nfhs4', 'district_name_nfhs5', 'status', 'district_code_nfhs5']
Stable districts in crosswalk: 575
IR4: 65884 rows (9.4%) unmapped to a stable district
IR5: 133650 rows (18.5%) unmapped to a stable district
IR4 stable districts represented: 575 / 575
IR5 stable districts represented: 575 / 575
Final scoped IR4: (633802, 10) | Final scoped IR5: (590465, 10)


In [19]:
# Diagnose the unmapped rows: are they concentrated (few large groups -> 
# possible join bug) or dispersed (many small groups -> genuine splits)?

ir4_unmapped_codes = ir4.loc[ir4["stable_district_id"].isna(), "district_code_raw"].value_counts()
ir5_unmapped_codes = ir5.loc[ir5["stable_district_id"].isna(), "district_code_raw"].value_counts()

print(f"IR4: {len(ir4_unmapped_codes)} distinct unmapped raw codes, "
      f"top 10 account for {ir4_unmapped_codes.head(10).sum() / ir4_unmapped_codes.sum():.1%} of unmapped rows")
print(ir4_unmapped_codes.head(10))
print()
print(f"IR5: {len(ir5_unmapped_codes)} distinct unmapped raw codes, "
      f"top 10 account for {ir5_unmapped_codes.head(10).sum() / ir5_unmapped_codes.sum():.1%} of unmapped rows")
print(ir5_unmapped_codes.head(10))

# Cross-check: does the crosswalk's own district_code_nfhs5 column contain
# these codes at all, or are they entirely absent from the crosswalk file?
cw_nfhs5_codes = set(crosswalk["district_code_nfhs5"].dropna())
print()
print(f"Of IR5's unmapped codes, {sum(c in cw_nfhs5_codes for c in ir5_unmapped_codes.index)} "
      f"/ {len(ir5_unmapped_codes)} DO appear somewhere in the crosswalk's district_code_nfhs5 column "
      f"(would indicate a mapping/dedup bug rather than a genuine gap)")

IR4: 65 distinct unmapped raw codes, top 10 account for 26.2% of unmapped rows
district_code_raw
135    2487
140    2281
409    2269
410    2251
289    1938
133    1251
294    1235
406    1232
293    1195
414    1153
Name: count, dtype: int64

IR5: 132 distinct unmapped raw codes, top 10 account for 9.7% of unmapped rows
district_code_raw
924    1357
928    1314
872    1311
929    1300
873    1283
930    1282
871    1280
925    1264
927    1258
921    1255
Name: count, dtype: int64

Of IR5's unmapped codes, 0 / 132 DO appear somewhere in the crosswalk's district_code_nfhs5 column (would indicate a mapping/dedup bug rather than a genuine gap)


In [20]:
# --- Segment 2: Birth-level construction ---

BR4_VARS = [c for c in ["caseid", "bidx", "b3", "b5", "b7"] if c in br4_cols]
BR5_VARS = [c for c in ["caseid", "bidx", "b3", "b5", "b7"] if c in br5_cols]

br4, _ = pyreadstat.read_dta(str(BR4_PATH), usecols=BR4_VARS, apply_value_formats=False)
br5, _ = pyreadstat.read_dta(str(BR5_PATH), usecols=BR5_VARS, apply_value_formats=False)

# Keep v011 (woman's DOB, CMC) alongside for the denominator segment later
ir4_keep = ir4_full[["caseid", "stable_district_id", "v005", "v008", "v011", "v025", "v106", "v190", "v104"]]
ir5_keep = ir5_full[["caseid", "stable_district_id", "v005", "v008", "v011", "v025", "v106", "v190", "v104"]]

br4 = br4.merge(ir4_keep, on="caseid", how="inner").assign(round="NFHS-4")
br5 = br5.merge(ir5_keep, on="caseid", how="inner").assign(round="NFHS-5")

births = pd.concat([br4, br5], ignore_index=True)
print("Full births shape (pre-exclusion):", births.shape)

# --- v104 cleaning (same logic as the pilot) ---
# 95 = "always lived here" -> infinite duration, never misattributed
# 96 = "visitor" -> no reliable residence tie, permanently excluded
n_visitors = (births["v104"] == 96).sum()
print(f"Dropping {n_visitors} birth records ({n_visitors/len(births):.1%}) tied to visitor-coded women")
births = births[births["v104"] != 96].copy()
births["v104_clean"] = births["v104"].replace({95: np.inf})

# --- Timing variables ---
births["birth_year"] = 1900 + (births["b3"] - 1) // 12
births["birth_month"] = (births["b3"] - 1) % 12 + 1
births["years_before_survey"] = ((births["v008"] - births["b3"]) / 12).apply(np.floor)
births["weight"] = births["v005"] / 1_000_000

neg_timing = (births["years_before_survey"] < 0).sum()
print(f"Records with negative years_before_survey (data anomaly check): {neg_timing}")

# --- Migration misattribution flag (retained as a column, not dropped) ---
births["possibly_misattributed"] = births["years_before_survey"] > births["v104_clean"]

# --- Window classification (flag, not filter -- per the earlier decision) ---
PRIMARY_MAX_YEARS = 7       # CONFIRM: 5, 6, or 7?
SENSITIVITY_MAX_YEARS = 15

def classify_window(y):
    if y <= PRIMARY_MAX_YEARS:
        return "primary"
    elif y <= SENSITIVITY_MAX_YEARS:
        return "sensitivity"
    else:
        return "excluded"

births["window"] = births["years_before_survey"].apply(classify_window)

print(births["window"].value_counts())
print(births.groupby("round")["window"].value_counts())
print("Final births shape:", births.shape)
births.head()

Full births shape (pre-exclusion): (2235254, 14)
Dropping 25917 birth records (1.2%) tied to visitor-coded women
Records with negative years_before_survey (data anomaly check): 0
window
excluded       798120
sensitivity    726721
primary        684496
Name: count, dtype: int64
round   window     
NFHS-4  excluded       409938
        sensitivity    393824
        primary        374681
NFHS-5  excluded       388182
        sensitivity    332897
        primary        309815
Name: count, dtype: int64
Final births shape: (2209337, 21)


,caseid,bidx,b3,b5,b7,stable_district_id,v005,v008,v011,v025,...,v190,v104,round,v104_clean,birth_year,birth_month,years_before_survey,weight,possibly_misattributed,window
0,01000101 02,1,1141,1,NaN,640.0,191760,1387,835,1,...,3,12,NFHS-4,12.0,1995,1,20.0,0.19176,True,excluded
1,01000101 02,2,1117,1,NaN,640.0,191760,1387,835,1,...,3,12,NFHS-4,12.0,1993,1,22.0,0.19176,True,excluded
2,01000101 02,3,1089,1,NaN,640.0,191760,1387,835,1,...,3,12,NFHS-4,12.0,1990,9,24.0,0.19176,True,excluded
3,01000109 01,1,1154,1,NaN,640.0,191760,1387,903,1,...,4,20,NFHS-4,20.0,1996,2,19.0,0.19176,False,excluded
4,01000109 01,2,1129,1,NaN,640.0,191760,1387,903,1,...,4,20,NFHS-4,20.0,1994,1,21.0,0.19176,True,excluded


In [21]:
# --- Segment 3: Age-banding for numerator, exposure denominator construction ---

AGE_BANDS = [(15,19),(20,24),(25,29),(30,34),(35,39),(40,44),(45,49)]

def age_to_band(age):
    for lo, hi in AGE_BANDS:
        if lo <= age <= hi:
            return f"{lo}-{hi}"
    return None  # outside reproductive age range

# --- Numerator: mother's age at each birth ---
births["woman_birth_year"] = 1900 + (births["v011"] - 1) // 12
births["mother_age_at_birth"] = births["birth_year"] - births["woman_birth_year"]
births["age_band"] = births["mother_age_at_birth"].apply(age_to_band)

out_of_range = births["age_band"].isna().sum()
print(f"Births with mother's age outside 15-49 at birth: {out_of_range} ({out_of_range/len(births):.1%})")

# --- Denominator: women-years of exposure by district x calendar year x age band ---
ir4_exp = ir4_full[["stable_district_id", "v005", "v008", "v011"]].assign(round="NFHS-4")
ir5_exp = ir5_full[["stable_district_id", "v005", "v008", "v011"]].assign(round="NFHS-5")
ir_all = pd.concat([ir4_exp, ir5_exp], ignore_index=True)

ir_all["interview_year"] = 1900 + (ir_all["v008"] - 1) // 12
ir_all["woman_birth_year"] = 1900 + (ir_all["v011"] - 1) // 12
ir_all["weight"] = ir_all["v005"] / 1_000_000

# Expand each woman across years_before_survey = 0..15 (matches the births'
# primary+sensitivity scope) to compute her age -- and hence eligibility --
# in each historical calendar year. This is memory-heavy (~1.2M women x 16
# rows =~ 19.6M rows before the age filter); if it's slow on your machine,
# this is the natural place to chunk by round.
n = len(ir_all)
t_range = np.arange(0, SENSITIVITY_MAX_YEARS + 1)  # 0..15
idx = np.repeat(np.arange(n), len(t_range))
t_tiled = np.tile(t_range, n)

exp = ir_all.iloc[idx].reset_index(drop=True)
exp["years_before_survey"] = t_tiled
exp["calendar_year"] = exp["interview_year"] - exp["years_before_survey"]
exp["age_in_year"] = exp["calendar_year"] - exp["woman_birth_year"]
exp["age_band"] = exp["age_in_year"].apply(age_to_band)
exp = exp[exp["age_band"].notna()].copy()
exp["window"] = exp["years_before_survey"].apply(classify_window)

print("Exposure rows (person-years, post age filter):", exp.shape)

exposure_denom = (
    exp.groupby(["round", "stable_district_id", "calendar_year", "age_band", "window"])["weight"]
    .sum()
    .reset_index(name="women_years_exposure")
)
print("Denominator table shape:", exposure_denom.shape)
exposure_denom.head()

Births with mother's age outside 15-49 at birth: 21774 (1.0%)
Exposure rows (person-years, post age filter): (14793013, 13)
Denominator table shape: (112699, 6)


,round,stable_district_id,calendar_year,age_band,window,women_years_exposure
0,NFHS-4,1.0,2001,15-19,sensitivity,46.489479
1,NFHS-4,1.0,2001,20-24,sensitivity,43.604912
2,NFHS-4,1.0,2001,25-29,sensitivity,32.196045
3,NFHS-4,1.0,2001,30-34,sensitivity,22.172582
4,NFHS-4,1.0,2001,35-39,sensitivity,2.981276


In [22]:
# --- Segment 3 patch: regroup exposure on years_before_survey (not calendar_year) ---
# Reuses `exp` already built in segment 3 -- no need to redo the expensive expansion
exposure_denom = (
    exp.groupby(["round", "stable_district_id", "years_before_survey", "age_band", "window"])["weight"]
    .sum()
    .reset_index(name="women_years_exposure")
)
print("Denominator table shape (by years_before_survey):", exposure_denom.shape)

# --- Segment 4 (fixed): merge on years_before_survey; no debug logging ---
numerator = (
    births.groupby(["round", "stable_district_id", "years_before_survey", "age_band", "window"])["weight"]
    .sum()
    .reset_index(name="weighted_births")
)
print("Numerator shape:", numerator.shape)

panel = exposure_denom.merge(
    numerator,
    on=["round", "stable_district_id", "years_before_survey", "age_band", "window"],
    how="outer"
)
panel["weighted_births"] = panel["weighted_births"].fillna(0)
panel["women_years_exposure"] = panel["women_years_exposure"].fillna(0)

RATEABLE_WINDOWS = {"primary", "sensitivity"}
rateable_mask = panel["window"].isin(RATEABLE_WINDOWS)
mismatch_rateable = ((panel["weighted_births"] > 0) & (panel["women_years_exposure"] == 0) & rateable_mask).sum()
print(f"Rows with births but zero exposure, WITHIN rateable windows (should be ~0): {mismatch_rateable}")

panel["asfr"] = np.where(panel["women_years_exposure"] > 0, panel["weighted_births"] / panel["women_years_exposure"], np.nan)
print("ASFR panel shape:", panel.shape)

def summarize_district_year(g):
    window = g.name[-1]  # groupby keys, not g's columns -- pandas excludes them from g itself
    total_births = g["weighted_births"].sum()
    if window not in RATEABLE_WINDOWS:
        return pd.Series({"TFR": np.nan, "total_weighted_births": total_births,
                           "total_women_years": np.nan, "n_age_bands_present": np.nan})
    total_exposure = g["women_years_exposure"].sum()
    tfr = np.nan if total_exposure == 0 else 5 * g["asfr"].dropna().sum()
    return pd.Series({"TFR": tfr, "total_weighted_births": total_births,
                       "total_women_years": total_exposure, "n_age_bands_present": g["asfr"].notna().sum()})

tfr_panel = panel.groupby(["round", "stable_district_id", "years_before_survey", "window"]).apply(summarize_district_year).reset_index()

# Approximate calendar-year label for display/plotting only -- NOT used for
# any merge logic. Anchored at each round's approximate median fieldwork
# year (NFHS-4 ~2016, NFHS-5 ~2020). CONFIRM these anchors, or we can
# compute exact per-district median interview year instead if you'd rather.
ROUND_ANCHOR_YEAR = {"NFHS-4": 2016, "NFHS-5": 2020}
tfr_panel["year_approx"] = tfr_panel["round"].map(ROUND_ANCHOR_YEAR) - tfr_panel["years_before_survey"]

MIN_CELL_N = 30
tfr_panel["low_cell_count"] = tfr_panel["total_weighted_births"] < MIN_CELL_N

print("TFR panel shape:", tfr_panel.shape)
rateable_tfr = tfr_panel[tfr_panel["window"].isin(RATEABLE_WINDOWS)]
print("Rateable-window TFR describe():")
print(rateable_tfr["TFR"].describe())
print(f"Share of rateable cells flagged low_cell_count: {rateable_tfr['low_cell_count'].mean():.1%}")
rateable_tfr.sort_values(["stable_district_id", "round", "years_before_survey"]).head(10)

Denominator table shape (by years_before_survey): (111232, 6)
Numerator shape: (144456, 6)
Rows with births but zero exposure, WITHIN rateable windows (should be ~0): 0
ASFR panel shape: (163253, 8)
TFR panel shape: (38852, 10)
Rateable-window TFR describe():
count    18400.000000
mean         2.559422
std          0.956700
min          0.563222
25%          1.858150
50%          2.337634
75%          3.080476
max          9.968813
Name: TFR, dtype: float64
Share of rateable cells flagged low_cell_count: 25.1%


,round,stable_district_id,years_before_survey,window,TFR,total_weighted_births,total_women_years,n_age_bands_present,year_approx,low_cell_count
0,NFHS-4,1.0,0.0,primary,1.987733,21.594220,329.190532,7.0,2016.0,True
1,NFHS-4,1.0,1.0,primary,2.584651,27.681652,325.078334,7.0,2015.0,True
2,NFHS-4,1.0,2.0,primary,3.052146,32.563377,312.859179,7.0,2014.0,False
3,NFHS-4,1.0,3.0,primary,2.889398,28.303687,300.659088,7.0,2013.0,True
4,NFHS-4,1.0,4.0,primary,2.915705,27.967961,287.028057,7.0,2012.0,True
5,NFHS-4,1.0,5.0,primary,3.377607,31.435284,272.300689,7.0,2011.0,False
6,NFHS-4,1.0,6.0,primary,3.635937,32.789613,259.541658,6.0,2010.0,False
7,NFHS-4,1.0,7.0,primary,3.657159,31.305178,244.771619,6.0,2009.0,False
8,NFHS-4,1.0,8.0,sensitivity,4.112839,32.284544,230.274930,6.0,2008.0,False
9,NFHS-4,1.0,9.0,sensitivity,4.365653,31.228200,219.052150,6.0,2007.0,False


In [23]:
# --- Segment 5: cross-round consistency, coverage, and final write ---

# --- Cross-round consistency check on the overlapping calendar years ---
overlap = rateable_tfr.pivot_table(
    index=["stable_district_id", "year_approx"], columns="round", values="TFR"
).dropna()
print(f"Overlapping district-years with a TFR estimate from BOTH rounds: {len(overlap)}")
corr = overlap["NFHS-4"].corr(overlap["NFHS-5"])
mad = (overlap["NFHS-4"] - overlap["NFHS-5"]).abs().mean()
print(f"Correlation between NFHS-4 and NFHS-5 retrospective TFR estimates: {corr:.3f}")
print(f"Mean absolute difference: {mad:.3f}")

# --- District coverage: how many of the 575 stable districts have at least
# one non-null primary-window TFR estimate, per round? ---
coverage = (
    rateable_tfr[rateable_tfr["window"] == "primary"]
    .groupby("round")["stable_district_id"].nunique()
)
print("\nDistricts with >=1 primary-window TFR estimate, by round:")
print(coverage)
print(f"(out of 575 stable districts)")

# --- Attach district names for readability (reference only, not a merge key change) ---
name_lookup = crosswalk[["district_code", "district_name_nfhs4", "district_name_nfhs5"]].drop_duplicates()
tfr_panel_out = tfr_panel.merge(name_lookup, left_on="stable_district_id", right_on="district_code", how="left")

# --- Write outputs ---
OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

detailed_path = OUT_DIR / "asfr_panel_full.parquet"   # age-band level, for later SDID-by-age-band if needed
tfr_path = OUT_DIR / "tfr_panel_full.parquet"          # district-year rollup, main analysis object

panel.to_parquet(detailed_path, index=False)
tfr_panel_out.to_parquet(tfr_path, index=False)

print(f"\nWrote {detailed_path} ({panel.shape})")
print(f"Wrote {tfr_path} ({tfr_panel_out.shape})")

Overlapping district-years with a TFR estimate from BOTH rounds: 6900
Correlation between NFHS-4 and NFHS-5 retrospective TFR estimates: 0.751
Mean absolute difference: 0.463

Districts with >=1 primary-window TFR estimate, by round:
round
NFHS-4    575
NFHS-5    575
Name: stable_district_id, dtype: int64
(out of 575 stable districts)

Wrote /Users/eknoorsandhu/DHS_India_Research/data/processed/asfr_panel_full.parquet ((163253, 8))
Wrote /Users/eknoorsandhu/DHS_India_Research/data/processed/tfr_panel_full.parquet ((38852, 13))
